In [ ]:
from datasets import load_dataset

ds = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [ ]:
print(ds['train'])
print(ds['train'].features)


Dataset({
    features: ['text', 'label'],
    num_rows: 25000
})
{'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


You can access individual examples like this:

In [ ]:
example_text = ds['train'][0]['text']
example_label = ds['train'][0]['label']

print(f"First review text: {example_text[:100]}...")
print(f"First review label: {example_label}")

First review text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it w...
First review label: 0


In [ ]:
import os

# Get the cache files for the 'train' split
train_cache_files = ds['train'].cache_files

if train_cache_files:
    # The actual data files are usually in subdirectories of these cache files
    # Let's try to infer the main directory for the dataset
    # A common pattern is that the first cache file path can reveal the base directory
    first_file_path = train_cache_files[0]['filename']
    # Get the directory of the first file
    dataset_base_dir = os.path.dirname(first_file_path)

    # The actual data is likely one or two levels above or in a specific subfolder related to the dataset name.
    # Let's try to navigate up to find a more general 'imdb' folder.
    # This might require some heuristics depending on how 'datasets' organizes it.
    # A safer bet is to show the full paths and let the user explore.

    print("Cached files for the 'train' split:")
    for entry in train_cache_files:
        print(entry['filename'])

    # Try to find a common parent directory for exploration
    if dataset_base_dir:
        print(f"\nPotential base directory for IMDB dataset files: {dataset_base_dir}")
        print("You can use `!ls -l " + dataset_base_dir + "` to explore its contents.")
else:
    print("No cache files found for the 'train' split. This might happen if the data is very small or loaded differently.")


Cached files for the 'train' split:
/root/.cache/huggingface/datasets/stanfordnlp___imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31/imdb-train.arrow

Potential base directory for IMDB dataset files: /root/.cache/huggingface/datasets/stanfordnlp___imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31
You can use `!ls -l /root/.cache/huggingface/datasets/stanfordnlp___imdb/plain_text/0.0.0/e6281661ce1c48d982bc483cf8a173c1bbeb5d31` to explore its contents.


##TOKENIZE THE TEXT OF THE DATASET:
To tokenize the text, we'll use a pre-trained tokenizer from the transformers library.

In [ ]:
# Install the transformers library if you haven't already
!pip install transformers

In [ ]:
from transformers import AutoTokenizer

# Load a pre-trained tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define a tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

# Apply the tokenization function to the dataset
tokenized_ds = ds.map(tokenize_function, batched=True)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Now, let's look at the structure of the tokenized dataset and an example of the tokenized text:

In [ ]:
print(tokenized_ds['train'][0])

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

Before moving to the transformer, we fix the HTML tags taht appear on the text so it does not affect the fina results.

In [ ]:
# A simple way to clean text before tokenizing it
clean_text = example_text.replace("<br />", " ")
print(f"Original text (first 100 chars): {example_text[:500]}...")
print(f"Cleaned text (first 100 chars): {clean_text[:500]}...")

Original text (first 100 chars): I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attent...
Cleaned text (first 100 chars): I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.  The plot is centered around a young Swedish drama student named Lena who wants to learn everything she 

Step 3: Feeding the Numbers into the AI Brain (The Transformer)

In [ ]:
from transformers import AutoModelForSequenceClassification

# We tell the model to expect 2 categories: 0 (Negative) or 1 (Positive)
model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2.We package our numbers into Tensors

In [ ]:
import torch

# Take the input_ids from the first example of the tokenized dataset
input_ids_from_example = tokenized_ds['train'][0]['input_ids']

# Turn them into a Tensor
# (We use [input_ids_from_example] to simulate passing a 'batch' of text)
example_inputs = torch.tensor([input_ids_from_example])

print("Example inputs tensor created successfully:")
print(example_inputs)

Example inputs tensor created successfully:
tensor([[  101,  1045, 12524,  1045,  2572,  8025,  1011,  3756,  2013,  2026,
          2678,  3573,  2138,  1997,  2035,  1996,  6704,  2008,  5129,  2009,
          2043,  2009,  2001,  2034,  2207,  1999,  3476,  1012,  1045,  2036,
          2657,  2008,  2012,  2034,  2009,  2001,  8243,  2011,  1057,  1012,
          1055,  1012,  8205,  2065,  2009,  2412,  2699,  2000,  4607,  2023,
          2406,  1010,  3568,  2108,  1037,  5470,  1997,  3152,  2641,  1000,
          6801,  1000,  1045,  2428,  2018,  2000,  2156,  2023,  2005,  2870,
          1012,  1026,  7987,  1013,  1028,  1026,  7987,  1013,  1028,  1996,
          5436,  2003,  8857,  2105,  1037,  2402,  4467,  3689,  3076,  2315,
         14229,  2040,  4122,  2000,  4553,  2673,  2016,  2064,  2055,  2166,
          1012,  1999,  3327,  2016,  4122,  2000,  3579,  2014,  3086,  2015,
          2000,  2437,  2070,  4066,  1997,  4516,  2006,  2054,  1996,  2779,
        

3. Push the Numbers Through the Brain.
Now, we pass those numbers into the model:


In [ ]:
# Pass the data into the model to get the raw scores (called logits)
with torch.no_grad(): # This tells Python: "We are just testing, don't update the brain yet"
    outputs = model(example_inputs)

# Pull out the raw scores
logits = outputs.logits
print("Raw Scores:", logits)

#it would only give me numbers are super close to zero, it means the AI is basically guessing at random right now.

# Look at the raw scores and pick the index of the highest number
#To make the code automatically pick the biggest number without you having to look at it, we use a function called argmax.
#This function acts like a finger pointing at the highest score.
prediction = torch.argmax(logits, dim=1).item()

print("The AI Predicts category:", prediction)



Raw Scores: tensor([[ 0.0183, -0.0050]])
The AI Predicts category: 0


Now we need to check how well the model performs across a larger group of phrases, (evaluation).Since training the Transformer to get smart takes a long time, here I write a quick evaluation script to test the model on a small batch of 3 different phrases from your dataset at once to see its "Report Card.


In [ ]:
# 1. Grab 3 samples from your dataset (ds is the dataset)
batch_texts = ds["train"]["text"][:3]
batch_labels = ds["train"]["label"][:3]

# 2. Tokenize all 3 at the same time
# (padding=True makes all sentences the same length so the AI can read them together)
inputs = tokenizer(batch_texts, truncation=True, padding=True, return_tensors="pt")

# 3. Feed them all into the AI brain
with torch.no_grad():
    outputs = model(**inputs)

# 4. Get the predictions for all 3
batch_predictions = torch.argmax(outputs.logits, dim=1).tolist()

# 5. Compare the AI guesses to the real answers!
print("AI Guesses: ", batch_predictions)
print("Real Answers:", batch_labels)

AI Guesses:  [0, 0, 0]
Real Answers: [0, 0, 0]


1. Here we get the Training Data ready for these models, we need a larger chunk of data so they can actually learn something (unlike the 3 sentences we used to test the Transformer). Here w grab 1,000 sentences for training and 500 for testing.

In [ ]:
# Grab text and labels from your loaded dataset
# Ensure a balanced subset of positive and negative samples for training and testing.
#because here i had the problem that the logistic regression model was only finding 0 (negative) and none of the posotive reviews of the ds
#To fix this, we need to ensure that the train_labels contain a mix of both 0s and 1s.

# Separate positive and negative samples
positive_train_samples = ds["train"].filter(lambda example: example["label"] == 1)
negative_train_samples = ds["train"].filter(lambda example: example["label"] == 0)

positive_test_samples = ds["test"].filter(lambda example: example["label"] == 1)
negative_test_samples = ds["test"].filter(lambda example: example["label"] == 0)

# Determine the number of samples needed for a balanced subset
num_train_samples_per_class = 500 # For a total of 1000 train samples
num_test_samples_per_class = 250  # For a total of 500 test samples

# Select a balanced subset, shuffle each part, and combine
train_texts_pos = positive_train_samples.shuffle(seed=42)["text"][:num_train_samples_per_class]
train_labels_pos = positive_train_samples.shuffle(seed=42)["label"][:num_train_samples_per_class]

train_texts_neg = negative_train_samples.shuffle(seed=42)["text"][:num_train_samples_per_class]
train_labels_neg = negative_train_samples.shuffle(seed=42)["label"][:num_train_samples_per_class]

test_texts_pos = positive_test_samples.shuffle(seed=42)["text"][:num_test_samples_per_class]
test_labels_pos = positive_test_samples.shuffle(seed=42)["label"][:num_test_samples_per_class]

test_texts_neg = negative_test_samples.shuffle(seed=42)["text"][:num_test_samples_per_class]
test_labels_neg = negative_test_samples.shuffle(seed=42)["label"][:num_test_samples_per_class]


# Combine them
train_texts = train_texts_pos + train_texts_neg
train_labels = train_labels_pos + train_labels_neg

test_texts = test_texts_pos + test_texts_neg
test_labels = test_labels_pos + test_labels_neg

# Final shuffle to mix positive and negative reviews
import random
combined_train_data = list(zip(train_texts, train_labels))
random.seed(42)
random.shuffle(combined_train_data)
train_texts, train_labels = zip(*combined_train_data) # Unzip

combined_test_data = list(zip(test_texts, test_labels))
random.seed(42)
random.shuffle(combined_test_data)
test_texts, test_labels = zip(*combined_test_data) # Unzip

# Convert tuples back to lists
train_texts = list(train_texts)
train_labels = list(train_labels)
test_texts = list(test_texts)
test_labels = list(test_labels)


print(f"Unique labels in train_labels: {set(train_labels)}")
print(f"Unique labels in test_labels: {set(test_labels)}")

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/25000 [00:00<?, ? examples/s]

Unique labels in train_labels: {0, 1}
Unique labels in test_labels: {0, 1}


2. TF-IDF (The Old School Tokenizer)Instead of using the Hugging Face AI tokenizer, TF-IDF just counts words. It looks at which words show up a lot in positive reviews (like "excellent") and negative reviews (like "waste").

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np # Import numpy

# Create the TF-IDF math machine
vectorizer = TfidfVectorizer(max_features=5000) # Keep the top 5000 most common words

# Turn your raw text into numbers
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

y_train = np.array(train_labels) # Convert to numpy array
y_test = np.array(test_labels)   # Convert to numpy array

# Add a print statement to verify the contents of y_train just before Logistic Regression
print(f"Unique labels in y_train before LR training: {set(y_train)}")

Unique labels in y_train before LR training: {np.int64(0), np.int64(1)}


Here, we have to re-vectorize:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Creamos un nuevo traductor de palabras a números
vectorizer = TfidfVectorizer(max_features=5000)

# 2. Traducimos tus listas mezcladas y balanceadas
X_train_balanced = vectorizer.fit_transform(train_texts)
X_test_balanced = vectorizer.transform(test_texts)

# 3. Guardamos tus etiquetas mezcladas en variables claras
y_train_balanced = train_labels
y_test_balanced = test_labels


 3. Train & Evaluate Logistic Regression:

In [ ]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

# 1. Start a timer to measure training time
start_time = time.time()

# 2. Create and train the model
lr_model = LogisticRegression()
lr_model.fit(X_train_balanced, y_train_balanced) #Here I make sure I use the balanced ds

train_time_lr = time.time() - start_time

# 3. Get predictions and calculate scores
lr_preds = lr_model.predict(X_test_balanced)  #Here I make sure I use the balanced ds
lr_accuracy = accuracy_score(y_test_balanced, lr_preds)
lr_f1 = f1_score(y_test_balanced, lr_preds)

print(f"Logistic Regression Done! Accuracy: {lr_accuracy:.2%}, F1: {lr_f1:.2f}, Time: {train_time_lr:.2f}s")

Logistic Regression Done! Accuracy: 81.60%, F1: 0.82, Time: 0.06s


4. Train & Evaluate SVM.

In [ ]:
from sklearn.svm import LinearSVC

# 1. Start a timer
start_time = time.time()

# 2. Create and train the model
svm_model = LinearSVC()
svm_model.fit(X_train_balanced, y_train_balanced)

train_time_svm = time.time() - start_time

# 3. Get predictions and calculate scores
svm_preds = svm_model.predict(X_test_balanced)
svm_accuracy = accuracy_score(y_test_balanced, svm_preds)
svm_f1 = f1_score(y_test_balanced, svm_preds)

print(f"SVM Done! Accuracy: {svm_accuracy:.2%}, F1: {svm_f1:.2f}, Time: {train_time_svm:.2f}s")


SVM Done! Accuracy: 80.80%, F1: 0.80, Time: 0.02s


##Training the Transformer:

Problem: The model had used a dataset that wasn't balanced with 1s and 0s (which is what we had done before) and had instead used the original dataset, which only contained 0s and no 1s. That's why the F1 score was 0 and the accuracy was 100%.



In [ ]:
from datasets import Dataset, DatasetDict

# 1. Package your perfectly balanced python lists into a Hugging Face Dataset format
balanced_ds = DatasetDict({
    "train": Dataset.from_dict({"text": train_texts, "label": train_labels}),
    "test": Dataset.from_dict({"text": test_texts, "label": test_labels})
})

# 2. Tokenize this exact balanced dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding=True)

tokenized_balanced_ds = balanced_ds.map(tokenize_function, batched=True)

# 3. Pull out the finished splits for your trainer
train_dataset = tokenized_balanced_ds["train"]
test_dataset = tokenized_balanced_ds["test"]


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

print("Training on the balanced dataset...")
trainer.train()

# Check the true final scores
eval_results = trainer.evaluate()
print(f"Balanced Transformer Done! Accuracy: {eval_results['eval_accuracy']:.2%}, F1: {eval_results['eval_f1']:.2f}")


Training on the balanced dataset...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.691017,0.682714,0.608000,0.581197
2,0.586960,0.553683,0.740000,0.745098
3,0.445174,0.484457,0.790000,0.796905


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.445174,0.484457,3,0.790000,0.796905


Balanced Transformer Done! Accuracy: 79.00%, F1: 0.80
